# topocorr example: Tapantí (Costa Rica, 9.8° N), near the equator

At this latitude the sun passes north of the tile in part of the year and south of it in the
rest. Per-image c then differs between the two sides, and one curve c(solar zenith) cannot follow
it. topocorr detects this (`model.sun_both_sides`): `correct` warns and uses per-image SCS+C (or
per-image C for `c-stack`), and `diagnose` warns and records `tc_sun_both_sides`.

This notebook

1. shows the sun on both sides of the tile and the illumination of east- and west-facing slopes
   through the Landsat 5 years (sun angles only: one small request);
2. fits the Landsat 5 stack model and shows the warnings;
3. maps the NIRv drift artefact with `diagnose` (optional).

**Earth Engine use** (Community tier: 150 EECU-hours a month):

| step | EECU-hours | output |
|---|---|---|
| sun angles of all images | < 0.01 | none |
| `fit`, Landsat 5 | not measured (interactive requests) | model JSON |
| `diagnose` map, Landsat 5 (optional) | about 1.7-3.9 (as at Kalam) | GeoTIFF, about 20 MB |

**Resumable.** A model file, a started task (`tasks.json`) or a downloaded file is never made
twice; re-run from the top after an interruption.

In [ ]:
# Settings
from pathlib import Path

PROJECT = "my-ee-project"
SITE = "tapanti"  # prefix of every file and task name
OUT = Path(f"{SITE}_outputs")
DRIVE_FOLDER = "topocorr_exports"
RCLONE_REMOTE = None
CENTER = (-83.7576, 9.7753)  # lon, lat
HALF_WIDTH_M = 12000
WRS = (15, 53)
MAX_CLOUD = 70  # a cloudy tropical site
INDEX = "nirv"
MAP_L5 = False  # the diagnose map (about 1.7-3.9 EECU-hours)

if PROJECT.strip() in ("", "my-ee-project"):
    raise ValueError(
        "Set PROJECT to your Earth Engine Cloud project id in this cell, then run again."
    )

In [ ]:
import json
import shutil
import subprocess
import time
from pathlib import Path

import ee
import numpy as np
from tqdm.auto import tqdm

import topocorr

ee.Initialize(project=PROJECT)
OUT.mkdir(parents=True, exist_ok=True)
TASKS = OUT / "tasks.json"  # every Earth Engine task started by this notebook, by name


def load_tasks():
    return json.loads(TASKS.read_text(encoding="utf-8")) if TASKS.exists() else {}


def start_once(name, make_task):
    """Start a batch task unless one with this name already ran or is running (resumable)."""
    tasks = load_tasks()
    if name in tasks and tasks[name]["state"] not in ("FAILED", "CANCELLED"):
        print(f"{name}: already started ({tasks[name]['state']}), skipped")
        return
    task = make_task()
    task.start()
    tasks[name] = {"id": task.id, "state": "READY"}
    TASKS.write_text(json.dumps(tasks, indent=1), encoding="utf-8")
    print(f"{name}: started task {task.id}")


def wait_for_tasks(poll=60):
    """Progress bar over the started tasks until all have finished; reports EECU-hours."""
    tasks = load_tasks()
    done_states = ("COMPLETED", "FAILED", "CANCELLED")
    with tqdm(total=len(tasks), desc="Earth Engine tasks", unit="task") as bar:
        while True:
            eecu = 0.0
            for t in tasks.values():
                s = ee.data.getTaskStatus(t["id"])[0]
                t["state"] = s["state"]
                t["eecu_hours"] = round((s.get("batch_eecu_usage_seconds") or 0) / 3600, 3)
                t["error"] = s.get("error_message", "")
                eecu += t["eecu_hours"]
            TASKS.write_text(json.dumps(tasks, indent=1), encoding="utf-8")
            finished = sum(t["state"] in done_states for t in tasks.values())
            bar.n = finished
            bar.set_postfix(eecu_h=f"{eecu:.2f}")
            bar.refresh()
            if finished == len(tasks):
                break
            time.sleep(poll)
    for name, t in tasks.items():
        print(f"{name}: {t['state']}, {t['eecu_hours']} EECU-h {t['error']}")


def fetch(files):
    """Copy finished exports from Drive with rclone (if RCLONE_REMOTE is set), else list them."""
    missing = [f for f in files if not (OUT / f).exists()]
    if not missing:
        print("all files are present in", OUT)
        return
    if RCLONE_REMOTE and shutil.which("rclone"):
        for f in tqdm(missing, desc="download", unit="file"):
            subprocess.run(["rclone", "copy", f"{RCLONE_REMOTE}{f}", str(OUT)], check=True)
    else:
        print(f"Download these files from the Google Drive folder {DRIVE_FOLDER!r} into {OUT}:")
        for f in missing:
            print("  ", f)


def scene_grid(images):
    """CRS and transform of the first image's 30 m grid: maps line up with the Landsat pixels."""
    p = ee.Image(images.first()).select("nir").projection().getInfo()
    return p["crs"], p["transform"]

In [ ]:
center = ee.Geometry.Point(CENTER)
utm = f"EPSG:{32600 + int((CENTER[0] + 180) // 6) + 1}"
x, y = center.transform(utm, 1).coordinates().getInfo()
x, y = round(x / 30) * 30, round(y / 30) * 30
aoi = ee.Geometry.Rectangle(
    [x - HALF_WIDTH_M, y - HALF_WIDTH_M, x + HALF_WIDTH_M, y + HALF_WIDTH_M],
    ee.Projection(utm),
    False,
)
l5 = (
    topocorr.collection("L5", aoi, "1990-01-01", "2013-01-01", max_cloud=MAX_CLOUD)
    .filter(ee.Filter.And(ee.Filter.eq("WRS_PATH", WRS[0]), ee.Filter.eq("WRS_ROW", WRS[1])))
    .sort("system:time_start")
)
print(l5.size().getInfo(), "Landsat 5 images")

## 1. The sun on both sides of the tile

In [ ]:
import eofigs
import matplotlib.pyplot as plt

from topocorr.stack import sun_sides

eofigs.set_style(journal="custom", font="Arial", bold=True, labels="A")


def sun_table(images):
    """Sun angles and dates of every image (one small request; no pixels)."""
    info = ee.Dictionary(
        {
            "time": images.aggregate_array("system:time_start"),
            "zenith": images.aggregate_array("tc_sun_zenith"),
            "azimuth": images.aggregate_array("tc_sun_azimuth"),
        }
    ).getInfo()
    years = 1970 + np.asarray(info["time"], float) / (365.25 * 86400e3)
    return years, np.asarray(info["zenith"], float), np.asarray(info["azimuth"], float)


def cos_i(zenith, azimuth, slope, aspect):
    z, a, s, e = (np.radians(v) for v in (zenith, azimuth, slope, aspect))
    return np.cos(z) * np.cos(s) + np.sin(z) * np.sin(s) * np.cos(a - e)


def illumination_by_year(series, slope=30.0, months=None, title=""):
    """Yearly median cos i of an east- and a west-facing slope, for each sensor series.

    ``months`` (e.g. (6, 7, 8)) keeps the same season every year, so the change between years
    is not hidden by a changing mix of acquisition dates.
    """
    fig, ax = eofigs.new_figure(width="single", height_mm=60)
    for label, (years, zen, azi) in series.items():
        if months is not None:
            keep = np.isin(np.floor((years % 1) * 12).astype(int) + 1, months)
            years, zen, azi = years[keep], zen[keep], azi[keep]
        year = np.floor(years).astype(int)
        for aspect, style in ((90.0, "-"), (270.0, "--")):
            ci = cos_i(zen, azi, slope, aspect)
            ys = np.unique(year)
            med = [np.median(ci[year == y]) for y in ys]
            side = "east" if aspect == 90.0 else "west"
            ax.plot(ys, med, style, marker="o", ms=2.5, label=f"{label}, {side}-facing")
    ax.set_xlabel("Year")
    season = "" if months is None else f", months {months[0]}-{months[-1]}"
    ax.set_ylabel(f"Median cos i ({slope:.0f}° slope{season})")
    ax.legend(fontsize=6, frameon=False)
    if title:
        ax.set_title(title)
    return fig


years, zen, azi = sun_table(l5)
north, south = sun_sides(azi)
print(f"sun north of the tile in {north} images, south of it in {south}")

fig = illumination_by_year({"Landsat 5": (years, zen, azi)})
eofigs.save_figure(fig, OUT / "fig_illumination_by_year")
plt.show()

## 2. Stack model and the warnings
Progress is printed as `images done / total`; an interrupted fit continues from its cache.

In [ ]:
import warnings

path = OUT / f"{SITE}_l5_model.json"
if path.exists():
    model = topocorr.load_model(path)
else:
    model = topocorr.fit(
        l5, strata="worldcover-density", region=aoi, cache=OUT / f"{SITE}_l5_fit_cache.json"
    )
    model.to_json(path)
print("sun on both sides:", model.sun_both_sides)

# Building the computations costs nothing; the warnings come from the model alone.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    corrected = topocorr.correct(l5, model)  # scsc-stack -> per-image SCS+C here
    risk = topocorr.diagnose(l5, model, index=INDEX)
for w in caught:
    print("warning:", w.message)

## 3. Drift map (optional)

In [ ]:
if MAP_L5:
    crs, transform = scene_grid(l5)
    key = f"{SITE}_diagnose_{INDEX}_l5"
    start_once(
        key,
        lambda: ee.batch.Export.image.toDrive(
            image=risk.toFloat(),
            description=key,
            folder=DRIVE_FOLDER,
            fileNamePrefix=key,
            region=aoi,
            crs=crs,
            crsTransform=transform,
            maxPixels=1e9,
        ),
    )
    wait_for_tasks()
    fetch([f"{key}.tif"])